In [ ]:
import numpy as np
import pandas as pd
import math

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn import metrics
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
import IPython
import graphviz
from sklearn.base import ClassifierMixin
from sklearn.tree import export_graphviz
import re

In [ ]:
def draw_tree(
    tree: ClassifierMixin, 
    feature_names: list, 
    size: int = 10, 
    ratio: float = 0.6, 
    precision: int = 0,
    ):
    """
    Draws a representation of a random forest in IPython.
    """
    # Las columnas son tuplas ('Open','BTC-USD'): las convertimos a strings
    names = ["_".join(map(str, f)).strip("_") if isinstance(f, tuple) else str(f)
             for f in feature_names]
    s = export_graphviz(
        tree, 
        out_file = None, 
        feature_names = names, 
        filled = True, 
        special_characters = True, 
        rotate = True, 
        precision = precision,
    )
    IPython.display.display(
        graphviz.Source(re.sub('Tree {', f'Tree {{ size={size}; ratio={ratio}', s))
    )

In [ ]:
def rmse(y_true, y_pred): 
    return math.sqrt(((y_true - y_pred)**2).mean())

In [ ]:
def print_score(m, X_train, y_train, X_valid, y_valid):
    print('RMSE on train set: {:.4f}'.format(rmse(m.predict(X_train), y_train)))
    print('RMSE on valid set: {:.4f}'.format(rmse(m.predict(X_valid), y_valid)))
    print('R^2 on train set: {:.4f}'.format(m.score(X_train, y_train)))
    print('R^2 on valid set: {:.4f}'.format(m.score(X_valid, y_valid)))
    if hasattr(m, 'oob_score_'): print('R^2 on oob set: {:.4f}'.format(m.oob_score_))
    return

### What we need to make this work
Okay so we need to have a df converted to a feather_file with such 
- all the features( parameters ) that we can get
- the y value we want to predict  

In [ ]:
df_treated = pd.read_feather('data/treated_dates.feather')
df_treated

In [ ]:
y = df_treated["up/down"]
df = df_treated.drop("up/down",axis=1)

In [ ]:
df


# time series split 

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

for train_index, test_index in tscv.split(df):
    X_train , X_test  = df.iloc[train_index] , df.iloc[test_index] 
    Y_train , Y_test   = y.iloc[train_index] , y.iloc[test_index]

In [ ]:
'''
for train_index, val_index in tscv.split(df):
    X_train, X_val = df.iloc[train_index], df.iloc[val_index]
    Y_train, Y_val = y.iloc[train_index], y.iloc[val_index]'''

In [ ]:
model = RandomForestRegressor(n_estimators= 10,
                              max_depth= 4,
                               min_samples_split= 10,
                                  n_jobs=-1,
                                    oob_score=True)

%time model.fit(X_train, Y_train)

In [ ]:
print_score(model, X_train , Y_train, X_test,Y_test)

In [ ]:
model.feature_importances_

In [ ]:
model.score(X_test,Y_test)

In [ ]:
model.estimators_[0]

In [ ]:
draw_tree(tree=model.estimators_[9], feature_names=X_train.columns.tolist(), precision = 3)